In [6]:
import numpy as np
import xarray as xr
from scipy.signal import detrend
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from sst_pix2pix_annual_cycle import (
    SSTDataset, Generator, compute_climatology, 
    rollout_prediction, get_seasonal_encoding
)
from train_sst_model import (
    Discriminator, CombinedLoss, 
    plot_training_history, plot_predictions
)

In [7]:
# ============================================================================
# CMIP6 Data Processing
# ============================================================================

def load_and_process_cmip6_data(dataset_path, mask_path='../../Data/mask.npy', 
                                 n_train=1428, n_val=100, ensemble_id=0):
    """
    Load and process CMIP6 SST data.
    
    Args:
        dataset_path: Path to CMIP6 netCDF file
        mask_path: Path to mask file
        n_train: Number of training samples
        n_val: Number of validation samples
        ensemble_id: Which ensemble member to use (if multiple)
    
    Returns:
        train_data, val_data, test_data, train_months, val_months, test_months, mask
    """
    print(f"Loading dataset from: {dataset_path}")
    
    # Load dataset
    ds = xr.open_dataset(dataset_path)
    
    # Select SST variable
    sst = ds["sst"]
    
    # If multiple ensembles, select one
    if "ensemble" in sst.dims:
        print(f"Multiple ensembles found. Selecting ensemble {ensemble_id}")
        sst = sst.isel(ensemble=ensemble_id)
    
    # Fill NaN values with 0
    sst = sst.fillna(0)
    
    sst = sst - sst.mean(dim="years")  # Center data by removing mean over time

    # Stack years and months into time dimension
    sst_time = (
        sst
        .stack(time=("years", "mon"))
        .transpose("time", "lat", "lon")
    ).fillna(0)
    
    data = sst_time.values  # Shape: (time, lat, lon)
    
    print(f"Data shape: {data.shape}")
    print(f"Data range: [{data.min():.2f}, {data.max():.2f}]")
    
    # Load mask
    mask = np.load(mask_path)
    print(f"Mask shape: {mask.shape}")
    
    # Split data
    train_data = data[:n_train, :, :]
    val_data = data[n_train:n_train + n_val, :, :]
    test_data = data[n_train:, :, :]
    
    print(f"Train shape: {train_data.shape}")
    print(f"Val shape: {val_data.shape}")
    print(f"Test shape: {test_data.shape}")
    
    # Create month arrays (0-11 for each sample)
    # Assuming data starts from a known month, or extract from dataset
    # If your dataset has months encoded, extract them. Otherwise:
    total_samples = data.shape[0]
    
    # Try to get starting month from dataset metadata
    if 'mon' in ds.coords:
        # If months are in coordinates, use them
        months_all = np.array([(i % 12) for i in range(total_samples)])
    else:
        # Default: assume data starts in January (month 0)
        months_all = np.array([i % 12 for i in range(total_samples)])
    
    train_months = months_all[:n_train]
    val_months = months_all[n_train:n_train + n_val]
    test_months = months_all[n_train:]
    
    return train_data, val_data, test_data, train_months, val_months, test_months, mask


def process_with_detrending(data, detrend_type='linear'):
    """
    Optional: Apply detrending to SST data.
    
    Args:
        data: SST data (time, lat, lon)
        detrend_type: 'linear' or 'constant'
    
    Returns:
        detrended data
    """
    if detrend_type is None:
        return data
    
    print(f"Applying {detrend_type} detrending...")
    data_detrended = np.zeros_like(data)
    
    # Detrend along time axis for each spatial point
    for i in range(data.shape[1]):
        for j in range(data.shape[2]):
            data_detrended[:, i, j] = detrend(data[:, i, j], type=detrend_type)
    
    return data_detrended


def normalize_data(train_data, val_data, test_data, method='standardize'):
    """
    Normalize SST data.
    
    Args:
        train_data, val_data, test_data: SST arrays
        method: 'standardize' (z-score) or 'minmax' (0-1 scaling)
    
    Returns:
        normalized data and normalization parameters
    """
    if method == 'standardize':
        mean = np.mean(train_data)
        std = np.std(train_data)
        
        train_norm = (train_data - mean) / std
        val_norm = (val_data - mean) / std
        test_norm = (test_data - mean) / std
        
        norm_params = {'mean': mean, 'std': std, 'method': 'standardize'}
        
        print(f"Standardization - Mean: {mean:.4f}, Std: {std:.4f}")
        
    elif method == 'minmax':
        min_val = np.min(train_data)
        max_val = np.max(train_data)
        
        train_norm = (train_data - min_val) / (max_val - min_val)
        val_norm = (val_data - min_val) / (max_val - min_val)
        test_norm = (test_data - min_val) / (max_val - min_val)
        
        norm_params = {'min': min_val, 'max': max_val, 'method': 'minmax'}
        
        print(f"Min-Max scaling - Min: {min_val:.4f}, Max: {max_val:.4f}")
    
    else:
        # No normalization
        train_norm, val_norm, test_norm = train_data, val_data, test_data
        norm_params = {'method': 'none'}
    
    return train_norm, val_norm, test_norm, norm_params


def denormalize_data(data, norm_params):
    """
    Denormalize data back to original scale.
    
    Args:
        data: normalized data
        norm_params: dictionary with normalization parameters
    
    Returns:
        denormalized data
    """
    if norm_params['method'] == 'standardize':
        return data * norm_params['std'] + norm_params['mean']
    elif norm_params['method'] == 'minmax':
        return data * (norm_params['max'] - norm_params['min']) + norm_params['min']
    else:
        return data


In [8]:
# ============================================================================
# COBE Data Processing
# ============================================================================

def load_and_process_cobe_data(dataset_path, mask_path='../../Data/mask.npy', 
                                 n_train=1428, n_val=100, ensemble_id=0):
    """
    Load and process COBE SST data.
    
    Args:
        dataset_path: Path to COBE netCDF file
        mask_path: Path to mask file
        n_train: Number of training samples
        n_val: Number of validation samples
        ensemble_id: Which ensemble member to use (if multiple)
    
    Returns:
        train_data, val_data, test_data, train_months, val_months, test_months, mask
    """
    print(f"Loading dataset from: {dataset_path}")
    
    # Load dataset
    ds = xr.open_dataset(dataset_path)
    
    # Select SST variable
    sst = ds["sst"]
    
    climatology = sst.groupby("time.month").mean(dim="time")
    sst = sst.groupby("time.month")-climatology
    sst = sst.fillna(0)

    data = sst.values  # Shape: (time, lat, lon)
    
    print(f"Data shape: {data.shape}")
    print(f"Data range: [{data.min():.2f}, {data.max():.2f}]")
    
    # Load mask
    mask = np.load(mask_path)
    print(f"Mask shape: {mask.shape}")
    
    # Split data
    train_data = data[:n_train, :, :]
    val_data = data[n_train:n_train + n_val, :, :]
    test_data = data[n_train:, :, :]
    
    print(f"Train shape: {train_data.shape}")
    print(f"Val shape: {val_data.shape}")
    print(f"Test shape: {test_data.shape}")
    
    # Create month arrays (0-11 for each sample)
    # Assuming data starts from a known month, or extract from dataset
    # If your dataset has months encoded, extract them. Otherwise:
    total_samples = data.shape[0]
    
    # Try to get starting month from dataset metadata
    if 'mon' in ds.coords:
        # If months are in coordinates, use them
        months_all = np.array([(i % 12) for i in range(total_samples)])
    else:
        # Default: assume data starts in January (month 0)
        months_all = np.array([i % 12 for i in range(total_samples)])
    
    train_months = months_all[:n_train]
    val_months = months_all[n_train:n_train + n_val]
    test_months = months_all[n_train:]
    
    return train_data, val_data, test_data, train_months, val_months, test_months, mask


In [9]:


def process_with_detrending(data, detrend_type='linear'):
    """
    Optional: Apply detrending to SST data.
    
    Args:
        data: SST data (time, lat, lon)
        detrend_type: 'linear' or 'constant'
    
    Returns:
        detrended data
    """
    if detrend_type is None:
        return data
    
    print(f"Applying {detrend_type} detrending...")
    data_detrended = np.zeros_like(data)
    
    # Detrend along time axis for each spatial point
    for i in range(data.shape[1]):
        for j in range(data.shape[2]):
            data_detrended[:, i, j] = detrend(data[:, i, j], type=detrend_type)
    
    return data_detrended


def normalize_data(train_data, val_data, test_data, method='standardize'):
    """
    Normalize SST data.
    
    Args:
        train_data, val_data, test_data: SST arrays
        method: 'standardize' (z-score) or 'minmax' (0-1 scaling)
    
    Returns:
        normalized data and normalization parameters
    """
    if method == 'standardize':
        mean = np.mean(train_data)
        std = np.std(train_data)
        
        train_norm = (train_data - mean) / std
        val_norm = (val_data - mean) / std
        test_norm = (test_data - mean) / std
        
        norm_params = {'mean': mean, 'std': std, 'method': 'standardize'}
        
        print(f"Standardization - Mean: {mean:.4f}, Std: {std:.4f}")
        
    elif method == 'minmax':
        min_val = np.min(train_data)
        max_val = np.max(train_data)
        
        train_norm = (train_data - min_val) / (max_val - min_val)
        val_norm = (val_data - min_val) / (max_val - min_val)
        test_norm = (test_data - min_val) / (max_val - min_val)
        
        norm_params = {'min': min_val, 'max': max_val, 'method': 'minmax'}
        
        print(f"Min-Max scaling - Min: {min_val:.4f}, Max: {max_val:.4f}")
    
    else:
        # No normalization
        train_norm, val_norm, test_norm = train_data, val_data, test_data
        norm_params = {'method': 'none'}
    
    return train_norm, val_norm, test_norm, norm_params


def denormalize_data(data, norm_params):
    """
    Denormalize data back to original scale.
    
    Args:
        data: normalized data
        norm_params: dictionary with normalization parameters
    
    Returns:
        denormalized data
    """
    if norm_params['method'] == 'standardize':
        return data * norm_params['std'] + norm_params['mean']
    elif norm_params['method'] == 'minmax':
        return data * (norm_params['max'] - norm_params['min']) + norm_params['min']
    else:
        return data


In [10]:
# ============================================================================
# Load Trained Model
# ============================================================================

def load_trained_model(checkpoint_path, device='cpu'):
    """
    Load trained generator from checkpoint.
    
    Args:
        checkpoint_path: Path to checkpoint file
        device: 'cuda' or 'cpu'
    
    Returns:
        generator, climatology, norm_params, num_input_months
    """
    print(f"Loading checkpoint from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    num_input_months = checkpoint['num_input_months']
    in_channels = num_input_months + 2
    
    generator = Generator(in_channels=in_channels, features=128, out_channels=1).to(device)
    generator.load_state_dict(checkpoint['generator_state_dict'])
    generator.eval()
    
    #climatology = checkpoint['climatology']
    norm_params = checkpoint['norm_params']
    
    print(f"Model loaded from epoch {checkpoint['epoch']}")
    if 'best_val_loss' in checkpoint:
        print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")
    
    return generator, norm_params, num_input_months



In [11]:
from ewc import EWC, EWCLoss

In [39]:
class LongTermVariabilityLoss(torch.nn.Module):
    """
    Combined loss function incorporating all strategies for long-term variability.
    """
    def __init__(self,  
                 lambda_l1=100.0,
                 ):
        super().__init__()
        self.l1_loss = torch.nn.L1Loss()
        self.lambda_l1 = lambda_l1
        
        # # Climatology loss
        # if climatology_data is not None:
        #     from sst_pix2pix_annual_cycle import ClimatologyLoss
        #     self.clim_loss = ClimatologyLoss(climatology_data, lambda_clim)
        #     self.use_clim_loss = True
        # else:
        #     self.use_clim_loss = False
        
        # # Variance-preserving loss
        # self.var_loss = VariancePreservingLoss(lambda_var)
        
        # # Temporal consistency loss
        # self.tc_loss = TemporalConsistencyLoss(lambda_tc)
    
    def forward(self, predicted, target):
        """
        Args:
            predicted: Predictions (batch, 1, H, W)
            target: Ground truth (batch, 1, H, W)
            target_months: Month indices (batch,)
            previous_sst: Previous month's SST for temporal consistency (batch, 1, H, W)
        
        Returns:
            total_loss, loss_dict
        """
        # Reconstruction loss
        l1 = self.l1_loss(predicted, target)
        
        # # Climatology loss
        # clim = 0.0
        # if self.use_clim_loss and target_months is not None:
        #     clim = self.clim_loss(predicted, target_months)
        
        # # Variance-preserving loss
        # var = self.var_loss(predicted, target)
        
        # # Temporal consistency loss
        # tc = 0.0
        # if previous_sst is not None:
        #     tc = self.tc_loss(predicted, previous_sst)
        
        # Total loss
        total_loss = self.lambda_l1 * l1 #+ clim + var + tc
        
        loss_dict = {
            'l1': l1.item(),
            # 'climatology': clim if isinstance(clim, float) else clim.item(),
            # 'variance': var.item(),
            # 'temporal_consistency': tc if isinstance(tc, float) else tc.item()
        }
        
        return total_loss, loss_dict




In [40]:
# def get_base_criterion(generator, prev_loader):
#     total_loss = 0
#     num_batches = 0
#     loss_dict = {}
#     with torch.no_grad():
#         for batch in prev_loader:
#             inputs = batch['input'].to(device)
#             targets = batch['target'].to(device)
#             outputs = generator(inputs)
#             loss = torch.nn.L1Loss()(outputs, targets)
#             total_loss += loss.item()
#             num_batches += 1
#     return total_loss / num_batches, loss_dict

In [ ]:
# ============================================================================
# Training with COBE Data
# ============================================================================



def train_cobe_model_ewc(dataset_path, mask_path='../Data/mask.npy',ewc=None,
                     prev_dataset_path=None, prev_mask_path=None,
                      n_train=1428, n_val=100, ensemble_id=0,
                      num_epochs=100, num_input_months=3, num_recent_months=3,
                      batch_size=16, lr=0.00002,lambda_ewc=5000.0,  # EWC penalty weight
                      lambda_clim=0.1,
                      normalize=True, apply_detrend=None,
                      checkpoint_dir='./checkpoints',load_weight_path=None):
    """
    Complete training pipeline for CMIP6 SST data.
    
    Args:
        dataset_path: Path to CMIP6 netCDF file
        mask_path: Path to mask file
        n_train: Number of training samples
        n_val: Number of validation samples
        ensemble_id: Which ensemble to use
        num_epochs: Training epochs
        num_input_months: Number of previous months as input
        batch_size: Batch size
        lr: Learning rate (typically lower than initial training)
        lambda_ewc: EWC penalty weight (higher = more preservation)
        lambda_clim: Climatology loss weight
        normalize: Whether to normalize data
        apply_detrend: 'linear', 'constant', or None
        checkpoint_dir: Directory to save checkpoints
    """
    import os
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}\n")
    
    # ===========================
    # 1. Load and Process Data
    # ===========================
    train_data, val_data, test_data, train_months, val_months, test_months, mask = \
        load_and_process_cobe_data(dataset_path, mask_path, n_train, n_val, ensemble_id)
    _, _, prev_test_data, _, _, prev_test_months, mask = \
        load_and_process_cmip6_data(prev_dataset_path, prev_mask_path, n_train, n_val, ensemble_id)
    
    # Optional detrending
    if apply_detrend:
        train_data = process_with_detrending(train_data, apply_detrend)
        val_data = process_with_detrending(val_data, apply_detrend)
        test_data = process_with_detrending(test_data, apply_detrend)
    
    # Normalize data
    if normalize:
        train_data, val_data, test_data, norm_params = normalize_data(
            train_data, val_data, test_data, method='standardize'
        )
    else:
        norm_params = {'method': 'none'}
    
    # # ===========================
    # # 2. Compute Climatology
    # # ===========================
    # print("\nComputing climatology from training data...")
    # climatology = compute_climatology(train_data, train_months)
    # print(f"Climatology shape: {climatology.shape}")
    
    # ===========================
    # 3. Create Datasets
    # ===========================
    print("\nCreating datasets...")
    train_dataset = SSTDataset(
        sst_data=train_data,
        months=train_months,
        num_input_months=num_input_months,
        
    )
    
    val_dataset = SSTDataset(
        sst_data=val_data,
        months=val_months,
        num_input_months=num_input_months,
        
    )
    
    prev_dataset = SSTDataset(
        sst_data=prev_test_data,
        months=prev_test_months,
        num_input_months=num_input_months,
        
    )

    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=4,
        pin_memory=True if device == 'cuda' else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True if device == 'cuda' else False
    )
    prev_loader = DataLoader(
        prev_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True if device == 'cuda' else False
    )
   
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    print(f"Previous samples: {len(prev_dataset)}")
    
    # ===========================
    # 4. Initialize Models
    # ===========================
    in_channels = num_input_months + 2  # SST channels + seasonal encoding
    print(f"\nInitializing models with {in_channels} input channels...")
    
    #generator = Generator(in_channels=in_channels, features=128, out_channels=1).to(device)
    #discriminator = Discriminator(in_channels=in_channels + 1).to(device)
    
    # Initialize weights
    # def init_weights(m):
    #     if isinstance(m, torch.nn.Conv2d) or isinstance(m, torch.nn.ConvTranspose2d):
    #         torch.nn.init.normal_(m.weight, 0.0, 0.02)
    #         if m.bias is not None:
    #             torch.nn.init.constant_(m.bias, 0)
    #     elif isinstance(m, torch.nn.BatchNorm2d):
    #         torch.nn.init.normal_(m.weight, 1.0, 0.02)
    #         torch.nn.init.constant_(m.bias, 0)
    
    # generator.apply(init_weights)
    # discriminator.apply(init_weights)
    device = 'cuda' #if torch.cuda.is_available() else 'cpu'

    # Load model
    generator, norm_params, num_input_months = load_trained_model(
        load_weight_path, device
    )
    
    # ===========================
    # 5. Setup Training
    # ===========================
    base_criterion = LongTermVariabilityLoss()
    #bce_loss = torch.nn.BCEWithLogitsLoss()
    criterion = EWCLoss(base_criterion, ewc, lambda_ewc=lambda_ewc)

    g_optimizer = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    #d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    
    g_scheduler = torch.optim.lr_scheduler.StepLR(g_optimizer, step_size=30, gamma=0.5)
    #d_scheduler = torch.optim.lr_scheduler.StepLR(d_optimizer, step_size=30, gamma=0.5)
    
    # Training history
    history = {
        'train_g_loss': [],
        #'train_d_loss': [],
        'train_l1_loss': [],
        #'train_clim_loss': [],
        'val_g_loss': [],
        'val_l1_loss': []
    }
    
    # ===========================
    # 6. Training Loop
    # ===========================
    print(f"\nStarting training for {num_epochs} epochs...")
    print("=" * 70)
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        # Train
        generator.train()
        #discriminator.train()
        
        epoch_g_loss = 0
        #epoch_d_loss = 0
        #epoch_l1_loss = 0
        #epoch_clim_loss = 0
        
        for batch_idx, batch in enumerate(train_loader):
            inputs = batch['input'].to(device)
            targets = batch['target'].to(device)
            target_months = batch['target_month'].to(device)
            # Previous SST for temporal consistency
            num_recent = num_recent_months

            previous_sst = inputs[:, num_recent-1:num_recent, :, :]
            
            # Train Discriminator
            #d_optimizer.zero_grad()
            
            # real_pair = torch.cat([inputs, targets], dim=1)
            # d_real = discriminator(real_pair)
            # real_labels = torch.ones_like(d_real) * 0.9
            # d_real_loss = bce_loss(d_real, real_labels)
            
            fake_outputs = generator(inputs)
            # fake_pair = torch.cat([inputs, fake_outputs.detach()], dim=1)
            # d_fake = discriminator(fake_pair)
            # fake_labels = torch.zeros_like(d_fake) + 0.1
            # d_fake_loss = bce_loss(d_fake, fake_labels)
            
            # d_loss = (d_real_loss + d_fake_loss) / 2
            # d_loss.backward()
            # d_optimizer.step()
            
            # Train Generator
            g_optimizer.zero_grad()
            
            # fake_outputs = generator(inputs)
            # fake_pair = torch.cat([inputs, fake_outputs], dim=1)
            # d_fake = discriminator(fake_pair)
            
            # real_labels = torch.ones_like(d_fake)
            # g_adv_loss = bce_loss(d_fake, real_labels)
            #print(f"Fake outputs : {fake_outputs}, Target : {targets}")
            #g_recon_loss = criterion(fake_outputs, targets)

            # Compute loss with EWC
            # print( fake_outputs.shape)
            # print( targets.shape)
            g_recon_loss, g_recon_loss_dict = criterion(generator, fake_outputs, targets)#, target_months, previous_sst
            

            
            g_loss = g_recon_loss
            g_loss.backward()
            g_optimizer.step()
            
            epoch_g_loss += g_loss.item()
            #epoch_l1_loss += g_recon_loss_dict['l1']
            epoch_ewc_loss = g_recon_loss_dict['ewc_penalty']
            # epoch_d_loss += d_loss.item()
            #epoch_l1_loss += loss_dict['l1']
            #epoch_clim_loss += loss_dict['climatology']
            
            if batch_idx % 50 == 0:
                print(f'Epoch {epoch+1}/{num_epochs} [{batch_idx}/{len(train_loader)}]')
        
        # Validation
        generator.eval()
        val_g_loss = 0
        # val_l1_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['input'].to(device)
                targets = batch['target'].to(device)
                #target_months = batch['target_month'].to(device)
                
                fake_outputs = generator(inputs)
                g_recon_loss, _ = criterion(generator, fake_outputs, targets)
                
                val_g_loss += g_recon_loss.item()
                #val_l1_loss += loss_dict['l1']
        
        # Update schedulers
        g_scheduler.step()
        # d_scheduler.step()
        
        # Record history
        history['train_g_loss'].append(epoch_g_loss / len(train_loader))
        #history['train_d_loss'].append(epoch_d_loss / len(train_loader))
        #history['train_l1_loss'].append(epoch_l1_loss / len(train_loader))
        #history['train_clim_loss'].append(epoch_clim_loss / len(train_loader))
        history['val_g_loss'].append(val_g_loss / len(val_loader))
        #history['val_l1_loss'].append(val_l1_loss / len(val_loader))
        
        avg_val_loss = val_g_loss / len(val_loader)
        
        print(f'\n{"="*70}')
        print(f'Epoch {epoch+1}/{num_epochs} Summary:')
        print(f'Train - G: {history["train_g_loss"][-1]:.4f} ')
        print(f'Val   - G: {avg_val_loss:.4f}')
        print(f'{"="*70}\n')
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                # 'discriminator_state_dict': discriminator.state_dict(),
                'g_optimizer_state_dict': g_optimizer.state_dict(),
                #'d_optimizer_state_dict': d_optimizer.state_dict(),
                'history': history,
                # 'climatology': climatology,
                'norm_params': norm_params,
                'num_input_months': num_input_months,
                'best_val_loss': best_val_loss
            }, os.path.join(checkpoint_dir, 'best_model.pth'))
            print(f"✓ Best model saved (val_loss: {best_val_loss:.4f})")
        
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                #'discriminator_state_dict': discriminator.state_dict(),
                'g_optimizer_state_dict': g_optimizer.state_dict(),
                #'d_optimizer_state_dict': d_optimizer.state_dict(),
                'history': history,
                #'climatology': climatology,
                'norm_params': norm_params,
                'num_input_months': num_input_months
            }, os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth'))
    
    # Plot training history
    plot_training_history_with_val(history, save_path='training_history.png')
    
    return generator, history, norm_params, test_data, test_months


In [62]:
def plot_training_history_with_val(history, save_path='training_history.png'):
    """Plot training and validation losses."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0, 0].plot(history['train_g_loss'], label='Train')
    axes[0, 0].plot(history['val_g_loss'], label='Val')
    axes[0, 0].set_title('Generator Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # axes[0, 1].plot(history['train_d_loss'])
    # axes[0, 1].set_title('Discriminator Loss (Train)')
    # axes[0, 1].set_xlabel('Epoch')
    # axes[0, 1].set_ylabel('Loss')
    # axes[0, 1].grid(True)
    
    axes[0, 1].plot(history['train_l1_loss'], label='Train')
    axes[0, 1].plot(history['val_l1_loss'], label='Val')
    axes[0, 1].set_title('L1 Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # axes[1, 1].plot(history['train_clim_loss'])
    # axes[1, 1].set_title('Climatology Loss (Train)')
    # axes[1, 1].set_xlabel('Epoch')
    # axes[1, 1].set_ylabel('Loss')
    # axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Training history saved to: {save_path}")
    plt.close()



In [63]:
from ewc import EWC, EWCLoss, save_fisher_information, analyze_fisher_importance, visualize_fisher_distribution


In [64]:
def compute_fisher_on_cmip6(checkpoint_path, 
                            dataset_path,
                            
                            mask_path,
                            batch_size=16,
                            n_train=1428,
                            n_val=100,
                            ensemble_id=0,
                            fisher_samples=500,
                            apply_detrend='linear',
                            normalize=True,
                            device='cpu'):
    """
    Compute Fisher Information Matrix on CMIP6 data.
    
    Args:
        checkpoint_path: Path to pre-trained model checkpoint
        dataset_path: Path to CMIP6 dataset
        mask_path: Path to mask file
        fisher_samples: Number of samples for Fisher estimation
        device: Computation device
    
    Returns:
        generator, ewc, config, climatology, norm_params
    """
    print("="*70)
    print("STEP 1: COMPUTING FISHER INFORMATION ON CMIP6")
    print("="*70)
    
    # Load checkpoint
    print("\nLoading pre-trained model...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    num_in_channels = checkpoint['num_input_months'] + 2
    
    
    # Initialize model
    generator = Generator(
        in_channels=num_in_channels,
        features=128,
        out_channels=1
    ).to(device)
    generator.load_state_dict(checkpoint['generator_state_dict'])
    
    print(f"Model loaded from epoch {checkpoint['epoch']}")
    
    # Load CMIP6 data
    print("\nLoading CMIP6 data...")
    from longterm_variability import normalize_with_trend
    
    # ===========================
    # 1. Load and Process Data
    # ===========================
    train_data, val_data, test_data, train_months, val_months, test_months, mask = \
        load_and_process_cmip6_data(dataset_path, mask_path, n_train, n_val, ensemble_id)
    
    # Optional detrending
    if apply_detrend:
        # train_data = process_with_detrending(train_data, apply_detrend)
        # val_data = process_with_detrending(val_data, apply_detrend)
        test_data = process_with_detrending(test_data, apply_detrend)
    
    # Normalize data
    if normalize:
        train_data, val_data, test_data, norm_params = normalize_data(
            train_data, val_data, test_data, method='standardize'
        )
    else:
        norm_params = {'method': 'none'}
    
    # Normalize
    norm_params = checkpoint['norm_params']
    #climatology = checkpoint['climatology']
    
    # if norm_params['method'] == 'trend_aware':
    #     train_norm = normalize_with_trend(train_data, norm_params)
    # else:
    #     train_norm = (train_data - norm_params['mean']) / norm_params['std']
    
    # Normalize climatology
    #climatology_norm = np.zeros_like(climatology)
    # for month in range(12):
    #     climatology_norm[month] = (climatology[month] - norm_params['mean']) / norm_params['std']
    
    # Create dataset
    print("\nCreating CMIP6 dataset for Fisher computation...")
    total_samples = n_train + len(val_data)
    # train_indices = np.arange(n_train)
    
    # cmip6_dataset = ExtendedContextDataset(
    #     sst_data=train_norm,
    #     months=train_months,
    #     sample_indices=train_indices,
    #     num_recent_months=checkpoint_path['num_recent_months'],
    #     num_annual_lag_months=checkpoint_path['num_annual_lag_months'],
    #     num_longterm_samples=checkpoint_path['num_longterm_samples'],
    #     climatology=climatology_norm,
    #     total_samples=total_samples
    # )

    prev_dataset = SSTDataset(
        sst_data=test_data,
        months=test_months,
        num_input_months=checkpoint['num_input_months'],
        
    )
    prev_loader = DataLoader(
        prev_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True if device == 'cuda' else False
    )
   
    # Compute Fisher Information
    print(f"\nComputing Fisher Information Matrix...")
    print(f"Using {fisher_samples} samples (out of {len(prev_loader)} available)")
    
    ewc = EWC(generator, prev_loader, device=device, 
             fisher_estimation_sample_size=fisher_samples)
    
    # Analyze Fisher
    analyze_fisher_importance(ewc, top_k=15)
    visualize_fisher_distribution(ewc, save_path='fisher_cmip6_distribution.png')
    
    return generator, ewc, checkpoint, norm_params, total_samples


In [ ]:
# ============================================================================
# Main Execution
# ============================================================================

device = 'cuda' if torch.cuda.is_available() else 'cpu'



if __name__ == "__main__":
    # Configuration
    # DATASET_PATH = '../../Data/CMIP6-SST/GISS-E2-1-H_historical_1850_2014.nc'
    # Alternative datasets:
    # DATASET_PATH = '../../Data/CMIP6-SST/MIROC6_historical_1850_2014.nc'
    # DATASET_PATH = '../../Data/CMIP6-SST/EC_Earth3_CC_historical_1850_2014.nc'
    DATASET_PATH = '../../Data/COBE_SST_resize.nc'
    FISHER_SAMPLES = 7  # Use all available samples for Fisher estimation
    NUM_EPOCHS = 100
    NUM_INPUT_MONTHS = 3
    BATCH_SIZE = 16
    LR = 0.00002
    LAMBDA_CLIM = 0.1
    MASK_PATH = '../../Data/mask.npy'
    N_TRAIN = 1428
    N_VAL = 100
    ENSEMBLE_ID = 0
    CMIP6_DATASET = '../../Data/CMIP6-SST/EC_Earth3_CC_historical_1850_2014.nc'
    CMIP6_CHECKPOINT = './checkpoints_EC_Earth3_CC_anomaly/checkpoint_epoch_200.pth'
    # Step 1: Compute Fisher on CMIP6
    generator, ewc, config, norm_params, total_samples = compute_fisher_on_cmip6(
        checkpoint_path=CMIP6_CHECKPOINT,
        dataset_path=CMIP6_DATASET,
        mask_path=MASK_PATH,
        batch_size=BATCH_SIZE,
        fisher_samples=FISHER_SAMPLES,
        device=device
    )

    
    
    # Run training
    generator, history, norm_params, test_data, test_months = \
        train_cobe_model_ewc(
            dataset_path=DATASET_PATH,
            mask_path=MASK_PATH,
            ewc=ewc,
            prev_dataset_path=CMIP6_DATASET,
            
            prev_mask_path=MASK_PATH,
            n_train=N_TRAIN,
            n_val=N_VAL,
            ensemble_id=ENSEMBLE_ID,
            num_epochs=NUM_EPOCHS,
            num_input_months=NUM_INPUT_MONTHS,
            num_recent_months=3,

            batch_size=BATCH_SIZE,
            lr=LR,
            lambda_ewc=5000.0,  # EWC penalty weight
            lambda_clim=LAMBDA_CLIM,
            normalize=True,
            apply_detrend='linear',  # Set to 'linear' if you want detrending
            checkpoint_dir='./checkpoints_COBE_anomaly2',
            load_weight_path=CMIP6_CHECKPOINT  # Load pre-trained weights
        )
    
    print("\nTraining complete!")
    print("Best model saved in: ./checkpoints_COBE_anomaly2/best_model.pth")


STEP 1: COMPUTING FISHER INFORMATION ON CMIP6

Loading pre-trained model...
Model loaded from epoch 199

Loading CMIP6 data...
Loading dataset from: ../../Data/CMIP6-SST/EC_Earth3_CC_historical_1850_2014.nc
Multiple ensembles found. Selecting ensemble 0
Data shape: (1980, 48, 144)
Data range: [-11.41, 10.61]
Mask shape: (48, 144)
Train shape: (1428, 48, 144)
Val shape: (100, 48, 144)
Test shape: (552, 48, 144)
Applying linear detrending...
Standardization - Mean: -0.0972, Std: 0.6548

Creating CMIP6 dataset for Fisher computation...

Computing Fisher Information Matrix...
Using 7 samples (out of 35 available)

COMPUTING FISHER INFORMATION MATRIX
Estimating Fisher from 7 samples...


0it [00:00, ?it/s]


Batch 1:


1it [00:00,  3.08it/s]


Fisher Statistics:
  Total parameters: 197,992,705
  Mean Fisher value: 0.000000
  Max Fisher value: 0.003758
  Samples used: 16

FISHER INFORMATION ANALYSIS

Top 15 Most Important Layers:
----------------------------------------------------------------------
 1. final_up.0                                             0.047937
 2. initial_down.0                                         0.002452
 3. initial_down.2                                         0.001038
 4. down4.conv.0                                           0.000853
 5. down1.conv.0                                           0.000716
 6. up4.conv.0                                             0.000615
 7. up1.conv.0                                             0.000583
 8. up2.conv.0                                             0.000383
 9. down3.conv.0                                           0.000330
10. down2.conv.0                                           0.000233
11. up4.conv.1                                             

Fisher distribution plot saved: fisher_cmip6_distribution.png
Using device: cuda

Loading dataset from: ../../Data/COBE_SST_resize.nc
Data shape: (2099, 48, 144)
Data range: [-8.31, 6.47]
Mask shape: (48, 144)
Train shape: (1428, 48, 144)
Val shape: (100, 48, 144)
Test shape: (671, 48, 144)
Loading dataset from: ../../Data/CMIP6-SST/EC_Earth3_CC_historical_1850_2014.nc
Multiple ensembles found. Selecting ensemble 0
Data shape: (1980, 48, 144)
Data range: [-11.41, 10.61]
Mask shape: (48, 144)
Train shape: (1428, 48, 144)
Val shape: (100, 48, 144)
Test shape: (552, 48, 144)
Applying linear detrending...
Applying linear detrending...
Applying linear detrending...
Standardization - Mean: -0.0000, Std: 0.2868

Creating datasets...
Training samples: 1425
Validation samples: 97
Previous samples: 549

Initializing models with 5 input channels...
Loading checkpoint from: ./checkpoints_EC_Earth3_CC_anomaly/checkpoint_epoch_200.pth
Model loaded from epoch 199

Starting training for 100 epochs...
